# Evo2 DNA Sequence Generation and Scoring

![Evo2 DNA Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/evo2/hero.png)

This notebook demonstrates examples of how to use both tools involving the Evo2 model. In the first section, we use `run_evo2_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling with KV caching for efficient generation. In the second section, we use `run_evo2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo2")
display_overview("evo2")
display_docs_section("evo2", "Background")

# Evo2

Evo2 is Arc Institute's genome-scale DNA language model for sequence generation and scoring. Trained on billions of nucleotides spanning prokaryotic and eukaryotic genomes, Evo2 performs autoregressive generation of DNA from prompts and scores sequences by log-likelihood. The tool supports local CUDA GPU execution with KV caching for efficient long generations.

## Background

DNA encodes the instructions for all cellular processes. Beyond protein-coding genes (~1.5% of the human genome), the vast majority of genomic sequence consists of regulatory elements, structural features, repetitive elements, and sequences of unknown function. The "grammar" of DNA -- the patterns that distinguish functional from non-functional sequence -- is complex and context-dependent.

Genome-scale language models like Evo2 learn these patterns by training on diverse genomic sequences across all domains of life. By predicting each nucleotide given preceding context, the model captures:

- **Codon usage patterns** and open reading frame structure in coding regions
- **Regulatory motifs** ([transcription factor](https://en.wikipedia.org/wiki/Transcription_factor) binding sites, [promoter](https://en.wikipedia.org/wiki/Promoter_(genetics)) elements, splice signals)
- **Compositional biases** ([GC content](https://en.wikipedia.org/wiki/GC-content), [CpG islands](https://en.wikipedia.org/wiki/CpG_island), repetitive elements)
- **Long-range dependencies** ([enhancer](https://en.wikipedia.org/wiki/Enhancer_(genetics))-promoter interactions, chromatin domain boundaries)
- **Cross-kingdom sequence grammar** (from viral genomes to mammalian chromosomes)

Evo2 uses byte-level tokenization (vocab_size=512), where each DNA base maps to its ASCII value (A=65, C=67, G=71, T=84, N=78). This allows the model to handle any genomic sequence without a specialized tokenizer.

## Available tools

In [2]:
display_available_tools("evo2")

- **`run_evo2_sample()`** — Sample DNA sequences using Evo2 language model
- **`run_evo2_score()`** — Score DNA sequences using Evo2 language model

### `run_evo2_sample`

This tool utilizes Evo2 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over genomic DNA. The `temperature` and `top_k` parameters control sampling diversity: lower values produce conservative, high-confidence extensions while higher values allow more exploratory generation.

In [3]:
from proto_tools import (
    Evo2SampleInput,
    Evo2SampleConfig,
    run_evo2_sample,
)

In [4]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_sample")

# Create an Evo2 input with one DNA starting prompt
inputs = Evo2SampleInput(prompts=["ATGCGTAAA"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `list[str]` | required | Prompt sequence(s) to condition generation on |

In [5]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_sample")

# Configure sampling parameters | see docs above for all fields
config = Evo2SampleConfig(
    max_new_tokens=200,
    temperature=0.8,
    top_k=4,
    device="cuda",
)

**Config** — `Evo2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `prepend_prompt` | `bool` | `True` | Include the input prompt at the start of each generated sequence |
| `temperature` | `float` | `1.0` | Softmax temperature for sampling; lower is more deterministic |
| `top_p` | `float` | `1.0` | Nucleus sampling cutoff over per-position token probabilities |
| `batch_size` | `int` | `1` | Prompts per GPU forward pass; raise for throughput, lower if OOM |
| `model_checkpoint` | `Literal['evo2_7b', 'evo2_20b', 'evo2_40b', 'evo2_7b_base', 'evo2_40b_base', 'evo2_1b_base', 'evo2_7b_262k', 'evo2_7b_microviridae']` | `'evo2_7b'` | Evo2 weights variant |
| `local_path` | `str \| None` | `None` | Override the default download with a local weights directory |
| `top_k` | `int` | `4` | Limit sampling to the top-k most probable tokens at each step |
| `max_new_tokens` | `int` | `32` | Maximum number of new tokens to generate per prompt |
| `cached_generation` | `bool` | `True` | Use the model's per-call KV cache during generation |
| `force_prompt_threshold` | `int \| None` | `None` | Tokens to prefill in parallel before switching to prompt forcing |
| `max_seqlen` | `int \| None` | `None` | Maximum sequence length the KV cache will be sized for |
| `skip_special_tokens` | `bool` | `False` | Filter EOS/PAD bytes from the detokenized output |
| `stop_at_eos` | `bool` | `True` | Stop generation when an EOS (id=0) token is sampled |
| `old_kv_cache` | `proto_tools.tools.causal_models.evo2.evo2_sample.Evo2KVCacheRef \| None` | `None` | Worker-local KV cache handle to use for continued generation |
| `return_kv_cache` | `bool` | `False` | Return worker-local KV cache handles for continued generation |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |

In [6]:
# Run the sampling function
sample_result = run_evo2_sample(inputs, config)

Running run_evo2_sample [00:00]

In [7]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")

**Output** — `Evo2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Generated sequences |
| `logits` | `list[Any] \| None` | `None` | Per-position logits for each generated sequence |
| `kv_caches` | `list[proto_tools.tools.causal_models.evo2.evo2_sample.Evo2KVCacheRef] \| None` | `None` | Worker-local KV cache handles for continued generation |

Prompt:       ATGCGTAAA
Generated:    ATGCGTAAATGTAAACAGAATTTGGAATATAGTATAGAAGCTTTTATGCCTGTTTTAATGCTGTAATCTTAATGCATCGA...
Total length: 209 nucleotides


### `run_evo2_score`

Evo2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each nucleotide is given its preceding context. The resulting perplexity provides a single-number summary of sequence naturalness: lower perplexity means the sequence more closely matches the statistical patterns in the training data. This is useful for ranking generated candidates, comparing wild-type and engineered variants, or filtering out sequences that deviate substantially from natural genomic patterns.

In [8]:
from proto_tools import (
    Evo2ScoringInput,
    Evo2ScoringConfig,
    run_evo2_score,
)

In [9]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_score")

# Score two short DNA sequences
score_inputs = Evo2ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Sequences to score |

In [10]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_score")

# Configure scoring | see docs above for all fields
score_config = Evo2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",
)

**Config** — `Evo2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `model_checkpoint` | `Literal['evo2_7b', 'evo2_20b', 'evo2_40b', 'evo2_7b_base', 'evo2_40b_base', 'evo2_1b_base', 'evo2_7b_262k', 'evo2_7b_microviridae']` | `'evo2_7b'` | Evo2 weights variant |
| `local_path` | `str \| None` | `None` | Override HuggingFace download with a local weights directory |
| `prepend_bos` | `bool` | `False` | Prepend a beginning-of-sequence token before scoring |

In [11]:
# Run the scoring function
score_result = run_evo2_score(score_inputs, score_config)

Running run_evo2_score [00:00]

In [12]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.causal_models.shared_data_models.CausalModelScoringMetrics]` | required | List of scoring outputs, one per input sequence |

Sequence 1: ATGCGTAAA
    Log-likelihood:     -10.975
    Avg log-likelihood: -1.372
    Perplexity:         3.943
Sequence 2: ATGCGTAAATTT
    Log-likelihood:     -14.729
    Avg log-likelihood: -1.339
    Perplexity:         3.815


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo2_sequences.fasta'}")

score_result.export(name="evo2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo2_scores.csv'}")

Generated sequences exported to example_output/evo2_sequences.fasta
Scores exported to example_output/evo2_scores.csv
